# ZeroBus Ingest Endpoint Validation

End-to-end validation of the dbxWearables ZeroBus AppKit gateway. Tests the health endpoint, sends sample NDJSON payloads for all 5 HealthKit record types, and verifies data lands in the bronze table.

**Deployed App:** `dev-dbxw-0bus-ingest` (dev target)  
**Bronze Table:** `{catalog}.{schema}.wearables_zerobus`

In [0]:
dbutils.widgets.text("app_base_url", "https://dev-dbxw-0bus-ingest-7474657291520070.aws.databricksapps.com", "App Base URL")
dbutils.widgets.text("catalog_use", "hls_fde_dev", "Catalog")
dbutils.widgets.text("schema_use", "dev_matthew_giglia_wearables", "Schema")

APP_BASE_URL = dbutils.widgets.get("app_base_url").rstrip("/")
CATALOG = dbutils.widgets.get("catalog_use")
SCHEMA = dbutils.widgets.get("schema_use")

print(f"App Base URL : {APP_BASE_URL}")
print(f"Catalog      : {CATALOG}")
print(f"Schema       : {SCHEMA}")

In [0]:
import requests
import json
import uuid
import textwrap
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Authenticate to Databricks App (audience-scoped token exchange) ───────
# Databricks Apps require an OAuth token scoped to the app's audience.
# From a notebook, this is a 3-step process:
#   1. Get the app's OAuth client ID via the SDK
#   2. Exchange the notebook's internal token for an audience-scoped token
#   3. Use the scoped token as a Bearer token in requests
# Ref: https://docs.databricks.com/aws/en/dev-tools/databricks-apps/connect-local/

# Step 1: Get the app's OAuth client ID
# URL format: https://<app-name>-<workspace-id>.<region>.databricksapps.com
hostname = APP_BASE_URL.split("//")[1].split(".")[0]
parts = hostname.rsplit("-", 1)
app_name = parts[0] if parts[-1].isdigit() else hostname
print(f"App name: {app_name}")

app_details = w.apps.get(app_name)
app_client_id = app_details.oauth2_app_client_id
print(f"App OAuth client ID: {app_client_id}")

# Step 2: Exchange notebook token for audience-scoped access token
notebook_token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

token_resp = requests.post(
    url=f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)
assert token_resp.status_code == 200, f"Token exchange failed ({token_resp.status_code}): {token_resp.text}"

audience_token = token_resp.json()["access_token"]
AUTH_HEADERS = {"Authorization": f"Bearer {audience_token}"}
print(f"Token exchange: \u2705 audience-scoped token acquired")

# Step 3: Define endpoints
HEALTH_URL = f"{APP_BASE_URL}/api/v1/healthkit/health"
INGEST_URL = f"{APP_BASE_URL}/api/v1/healthkit/ingest"
TABLE_FQN  = f"{CATALOG}.{SCHEMA}.wearables_zerobus"

print(f"\nHealth URL : {HEALTH_URL}")
print(f"Ingest URL : {INGEST_URL}")
print(f"Table FQN  : {TABLE_FQN}")

## 1. Health Check

In [0]:
resp = requests.get(HEALTH_URL, headers=AUTH_HEADERS, timeout=15)

print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
assert resp.json()["env_configured"] is True, "env_configured is not True — ZeroBus env vars missing"

print("\n\u2705 Health check passed — all env vars configured")

## 2. Sample NDJSON Payloads

Payloads match the iOS HealthKit app's Codable models exactly (snake_case JSON keys via CodingKeys). Each record type sends NDJSON (one JSON object per line).

In [0]:
PAYLOADS = {
    "samples": [
        {
            "uuid": "A1B2C3D4-E5F6-7890-ABCD-EF1234567890",
            "type": "HKQuantityTypeIdentifierHeartRate",
            "value": 72.0,
            "unit": "count/min",
            "start_date": "2026-04-17T10:30:00Z",
            "end_date": "2026-04-17T10:30:00Z",
            "source_name": "Apple Watch",
            "source_bundle_id": "com.apple.health.D8A7B5C3-1234-5678-9ABC-DEF012345678",
            "metadata": {"HKMetadataKeyHeartRateMotionContext": "1"},
        },
        {
            "uuid": "B2C3D4E5-F6A7-8901-BCDE-F12345678901",
            "type": "HKQuantityTypeIdentifierStepCount",
            "value": 1247.0,
            "unit": "count",
            "start_date": "2026-04-17T08:00:00Z",
            "end_date": "2026-04-17T09:00:00Z",
            "source_name": "iPhone",
            "source_bundle_id": "com.apple.health",
            "metadata": None,
        },
    ],
    "workouts": [
        {
            "uuid": "C3D4E5F6-A7B8-9012-CDEF-123456789012",
            "activity_type": "running",
            "activity_type_raw": 37,
            "start_date": "2026-04-17T06:30:00Z",
            "end_date": "2026-04-17T07:05:00Z",
            "duration_seconds": 2100.0,
            "total_energy_burned_kcal": 320.5,
            "total_distance_meters": 5230.0,
            "source_name": "Apple Watch",
            "metadata": None,
        },
    ],
    "sleep": [
        {
            "start_date": "2026-04-16T22:30:00Z",
            "end_date": "2026-04-17T06:15:00Z",
            "stages": [
                {"uuid": "D4E5F6A7-B8C9-0123-DEF0-123456789ABC", "stage": "inBed",       "start_date": "2026-04-16T22:30:00Z", "end_date": "2026-04-17T06:15:00Z"},
                {"uuid": "E5F6A7B8-C9D0-1234-EF01-23456789ABCD", "stage": "asleepCore",  "start_date": "2026-04-16T23:00:00Z", "end_date": "2026-04-17T01:30:00Z"},
                {"uuid": "F6A7B8C9-D0E1-2345-F012-3456789ABCDE", "stage": "asleepDeep",  "start_date": "2026-04-17T01:30:00Z", "end_date": "2026-04-17T03:00:00Z"},
                {"uuid": "A7B8C9D0-E1F2-3456-0123-456789ABCDEF", "stage": "asleepREM",   "start_date": "2026-04-17T03:00:00Z", "end_date": "2026-04-17T04:00:00Z"},
            ],
        },
    ],
    "activity_summaries": [
        {
            "date": "2026-04-17",
            "active_energy_burned_kcal": 485.3,
            "active_energy_burned_goal_kcal": 500.0,
            "exercise_minutes": 32.0,
            "exercise_minutes_goal": 30.0,
            "stand_hours": 10,
            "stand_hours_goal": 12,
        },
    ],
    "deletes": [
        {
            "uuid": "A1B2C3D4-E5F6-7890-ABCD-EF1234567890",
            "sample_type": "HKQuantityTypeIdentifierHeartRate",
        },
    ],
}


def to_ndjson(records: list) -> str:
    """Serialize a list of dicts to NDJSON (one JSON object per line)."""
    return "\n".join(json.dumps(r) for r in records)


print(f"Defined {len(PAYLOADS)} record types: {', '.join(PAYLOADS.keys())}")

## 3. Ingest Test — POST /api/v1/healthkit/ingest

In [0]:
from datetime import datetime, timezone

results = {}
total_ingested = 0
failures = []

for record_type, records in PAYLOADS.items():
    ndjson_body = to_ndjson(records)
    headers = {
        **AUTH_HEADERS,
        # application/x-ndjson is the standard NDJSON content type (iOS app uses this).
        # The server's error-recovery middleware catches express.json() parse
        # failures for this content type and recovers the raw body.
        "Content-Type": "application/x-ndjson",
        "X-Record-Type": record_type,
        # Metadata headers matching iOS APIService.buildRequestHeaders()
        "X-Platform": "apple_healthkit",
        "X-App-Version": "1.0.0",
        "X-Upload-Timestamp": datetime.now(timezone.utc).isoformat(),
        "X-Device-Id": "validation-notebook",
    }

    resp = requests.post(INGEST_URL, headers=headers, data=ndjson_body, timeout=30)

    print(f"\n── {record_type} (sent {len(records)} record(s)) ──")
    print(f"Status: {resp.status_code}")

    try:
        resp_json = resp.json()
        print(json.dumps(resp_json, indent=2))
    except Exception:
        print(f"Response body (raw): {resp.text[:500]}")
        resp_json = {}

    results[record_type] = resp_json

    if resp.status_code == 200:
        total_ingested += resp_json.get("records_ingested", 0)
    else:
        failures.append(f"{record_type}: {resp.status_code}")

if failures:
    print(f"\n❌ {len(failures)} record type(s) failed: {', '.join(failures)}")
    raise AssertionError(f"Ingest failed for: {', '.join(failures)}")
else:
    print(f"\n✅ All {len(PAYLOADS)} record types ingested successfully — {total_ingested} total records")

## 4. Verify Data in Bronze Table

In [0]:
df = spark.sql(f"""
    SELECT record_id, ingested_at, record_type, body, headers
    FROM {TABLE_FQN}
    ORDER BY ingested_at DESC
    LIMIT 20
""")

row_count = df.count()
print(f"Rows returned: {row_count}")
display(df)

In [0]:
df_counts = spark.sql(f"""
    SELECT record_type, COUNT(*) AS count
    FROM {TABLE_FQN}
    GROUP BY record_type
    ORDER BY record_type
""")

display(df_counts)

total_expected = sum(len(recs) for recs in PAYLOADS.values())
total_actual = df_counts.groupBy().sum("count").collect()[0][0]
print(f"\nTotal records in table: {total_actual}")
print(f"Expected at least:     {total_expected}")
assert total_actual >= total_expected, (
    f"Expected at least {total_expected} records, found {total_actual}"
)
print(f"\u2705 Record count verified — {total_actual} rows across {df_counts.count()} record types")

In [0]:
df_sample = spark.sql(f"""
    SELECT
        record_id,
        record_type,
        body::STRING  AS body_json,
        headers::STRING AS headers_json
    FROM {TABLE_FQN}
    WHERE record_type = 'samples'
    ORDER BY ingested_at DESC
    LIMIT 1
""")

display(df_sample)

# Pretty-print the VARIANT body to verify NDJSON structure is intact
row = df_sample.collect()[0]
parsed_body = json.loads(row["body_json"])

print(f"\n── Parsed VARIANT body (record_id: {row['record_id']}) ──")
print(json.dumps(parsed_body, indent=2))
print(f"\n\u2705 VARIANT body round-trips correctly — {len(parsed_body)} top-level keys")

## 5. Bronze Table Schema Review

Cross-reference the iOS HealthKit app models, server header capture logic, and bronze table schema to validate completeness.

In [0]:
# ── Cross-reference: iOS headers sent vs server headers captured ──────────

# iOS APIService.swift buildRequestHeaders() sends:
ios_headers_sent = {
    "Content-Type":        "application/x-ndjson",
    "X-Device-Id":         "DeviceIdentifier.current",
    "X-Platform":          "apple_healthkit",
    "X-App-Version":       "CFBundleShortVersionString",
    "X-Upload-Timestamp":  "ISO8601 date string",
    "X-Record-Type":       "samples|workouts|sleep|activity_summaries|deletes",
    "Authorization":       "Bearer <token>  (excluded by design)",
}

# Server ingest-routes.ts HEADERS_TO_KEEP captures:
server_headers_captured = [
    "x-record-type",
    "x-device-id",
    "x-sync-session-id",   # placeholder — iOS doesn't send this yet
    "x-batch-index",       # placeholder — iOS doesn't send this yet
    "x-batch-count",       # placeholder — iOS doesn't send this yet
    "content-type",
    "user-agent",
]

# Normalize to lowercase for comparison
ios_keys = {k.lower() for k in ios_headers_sent if k != "Authorization"}
server_keys = set(server_headers_captured)

captured = ios_keys & server_keys
dropped  = ios_keys - server_keys  # iOS sends, server ignores
placeholders = server_keys - ios_keys  # Server expects, iOS doesn't send

print("═" * 70)
print("HEADER CAPTURE ANALYSIS")
print("═" * 70)

print(f"\n✅ Captured ({len(captured)}):")
for h in sorted(captured):
    print(f"   {h}")

print(f"\n❌ DROPPED — iOS sends but server ignores ({len(dropped)}):")
for h in sorted(dropped):
    desc = ios_headers_sent.get(h) or ios_headers_sent.get(h.title().replace('-', '-'))
    # Find the original case key
    for k, v in ios_headers_sent.items():
        if k.lower() == h:
            desc = v
            break
    print(f"   {h:30s} → {desc}")

print(f"\n⏳ Placeholders — server captures but iOS doesn't send yet ({len(placeholders)}):")
for h in sorted(placeholders):
    print(f"   {h}")

print("\n" + "═" * 70)
print("VERDICT: Add x-platform, x-app-version, x-upload-timestamp")
print("         to HEADERS_TO_KEEP in ingest-routes.ts")
print("═" * 70)

In [0]:
# ── Verify all 5 iOS record types map cleanly to VARIANT body ────────────

import json

# Define the canonical iOS model schemas (from healthKit/Models/*.swift)
ios_model_schemas = {
    "samples (HealthSample)": [
        "uuid", "type", "value", "unit", "start_date", "end_date",
        "source_name", "source_bundle_id", "metadata",
    ],
    "workouts (WorkoutRecord)": [
        "uuid", "activity_type", "activity_type_raw", "start_date",
        "end_date", "duration_seconds", "total_energy_burned_kcal",
        "total_distance_meters", "source_name", "metadata",
    ],
    "sleep (SleepRecord)": [
        "start_date", "end_date",
        "stages[].uuid", "stages[].stage", "stages[].start_date", "stages[].end_date",
    ],
    "activity_summaries (ActivitySummary)": [
        "date", "active_energy_burned_kcal", "active_energy_burned_goal_kcal",
        "exercise_minutes", "exercise_minutes_goal",
        "stand_hours", "stand_hours_goal",
    ],
    "deletes (DeletionRecord)": [
        "uuid", "sample_type",
    ],
}

print("═" * 70)
print("iOS MODEL → BRONZE TABLE MAPPING")
print("═" * 70)
print()
print("Table: hls_fde_dev.dev_matthew_giglia_wearables.wearables_zerobus")
print()
print("┌─────────────────┬────────────┬─────────────────────────────────────────┐")
print("│ Bronze Column    │ Type       │ Source                                  │")
print("├─────────────────┼────────────┼─────────────────────────────────────────┤")
print("│ record_id       │ STRING PK  │ Server: crypto.randomUUID()             │")
print("│ ingested_at     │ TIMESTAMP  │ Server: Date.now() * 1000 (μs)         │")
print("│ body            │ VARIANT    │ iOS: full NDJSON line (see models ↓)   │")
print("│ headers         │ VARIANT    │ iOS: curated HTTP headers               │")
print("│ record_type     │ STRING     │ iOS: X-Record-Type header              │")
print("└─────────────────┴────────────┴─────────────────────────────────────────┘")
print()

for model_name, fields in ios_model_schemas.items():
    print(f"  📱 {model_name}")
    for f in fields:
        print(f"       body:{f}")
    print()

print("─" * 70)
print("All 5 record types stored as-is in body VARIANT. No data lost.")
print("VARIANT preserves nested structures (sleep stages) and optionals.")
print("Silver layer will parse body per record_type — bronze stays raw.")

In [0]:
# ── Should we add an offset column? ──────────────────────────────────────
#
# Answer: NO — Delta row tracking already provides this.

print("═" * 70)
print("OFFSET COLUMN ANALYSIS")
print("═" * 70)

print("""
❓ Question: Add an auto-incrementing integer offset column?

❌ RECOMMENDATION: No — unnecessary given existing Delta features.

── Why Delta row tracking is sufficient ──────────────────────────────

The table already has:
  • delta.enableRowTracking = true
  • _row_id          — monotonically increasing per file (system column)
  • _row_commit_version — increments per commit (system column)

These provide:
  ✅ Global ordering across commits (_row_commit_version)
  ✅ Per-commit ordering within a batch (_row_id)
  ✅ Zero application-level coordination needed
  ✅ Works with concurrent requests automatically

── Why a user-space offset would be problematic ──────────────────────

  1. Concurrency hazard: Multiple simultaneous iOS sync POSTs would
     need coordinated MAX(offset)+1 queries — race conditions.
  2. ZeroBus is REST-based, not Kafka — no native stream offset concept.
  3. SDP downstream doesn't need it — checkpoints track by commit
     version, not by a user-space sequence number.
  4. ingested_at (μs precision) already orders records temporally.

── When you WOULD want an offset ─────────────────────────────────────

  • External consumer coordination (Kafka-style exactly-once)
  • Multi-consumer partition assignment
  • Neither applies here — SDP handles this via Delta checkpointing.

── How to query ordering without an offset column ────────────────────

  -- Within a batch (same commit):
  SELECT * FROM wearables_zerobus
  ORDER BY ingested_at, record_id

  -- Across batches (global order):
  SELECT *, _metadata.row_commit_version, _metadata.row_id
  FROM wearables_zerobus
  ORDER BY _metadata.row_commit_version, _metadata.row_id
""")

In [0]:
# ── Best Practices Audit ─────────────────────────────────────────────────

# Check current Delta table properties
df_props = spark.sql(f"""
    DESCRIBE DETAIL {TABLE_FQN}
""")

print("═" * 70)
print("BEST PRACTICES AUDIT")
print("═" * 70)

checks = [
    ("CDF enabled",              "delta.enableChangeDataFeed",     "true",  "SDP downstream reads changes"),
    ("Row tracking",             "delta.enableRowTracking",        "true",  "Monotonic ordering, replaces offset"),
    ("Variant shredding",        "delta.enableVariantShredding",   "true",  "Query perf on body/headers VARIANT"),
    ("Deletion vectors",         "delta.enableDeletionVectors",    "true",  "Efficient soft deletes"),
    ("ZSTD compression",         "delta.parquet.compression.codec","zstd",  "Best compression ratio"),
    ("Liquid clustering",        "clusterByAuto",                  "true",  "Auto-tuned data layout"),
    ("Predictive optimization",  None,                             None,    "Inherited from metastore"),
]

print()
for name, prop, expected, purpose in checks:
    if prop:
        actual = spark.sql(f"SHOW TBLPROPERTIES {TABLE_FQN} ('{prop}')").collect()
        val = actual[0][1] if actual else "NOT SET"
        status = "✅" if val == expected else "⚠️"
    else:
        status = "✅"
        val = "inherited"
    print(f"  {status} {name:30s} = {val:10s}  ({purpose})")

print()
print("─" * 70)
print("RECOMMENDATIONS")
print("─" * 70)
print("""
  🔧 FIX (server-side bug):

  1. Add missing headers to HEADERS_TO_KEEP in ingest-routes.ts:
     + 'x-platform'           — identifies data source (apple_healthkit, android, fitbit…)
     + 'x-app-version'        — critical for debugging payload format changes
     + 'x-upload-timestamp'   — client-side timestamp for latency measurement

  💡 CONSIDER for future multi-platform:

  2. Add `source_platform STRING` top-level column (denormalized from
     headers::"x-platform") — fast filtering by platform without
     parsing VARIANT. Similar to how record_type was denormalized.

  3. Add NOT NULL constraint on `record_type` — the server already
     rejects requests without a valid X-Record-Type (returns 400),
     so the column should enforce this at the table level too.

  4. Consider explicit clustering: CLUSTER BY (record_type, ingested_at)
     instead of CLUSTER BY AUTO. The table is new with few queries,
     so AUTO may not have enough signal yet. Once query patterns emerge
     from SDP and dashboards, AUTO will adapt.

  5. When iOS adds X-Sync-Session-Id (per SyncCoordinator sync cycle),
     consider denormalizing it too — enables grouping all records from
     a single sync operation.

  ✅ NO ACTION NEEDED:

  • Offset column — Delta row tracking handles this
  • Table schema (5 columns) — correct for bronze layer
  • VARIANT for body — preserves all 5 record types faithfully
  • VARIANT for headers — flexible for header evolution
  • Primary key on record_id — good for dedup and joins
""")